# Gemma 4 E4B baseline on Kaggle

Run the **untuned E4B competition checkpoint** against three public practice issues. The competition evaluator generates a patch and independently verifies it. This run establishes a baseline for a later LoRA comparison.

## Set up the Kaggle notebook

1. Import this .ipynb into a Kaggle notebook linked to the [Gemma 4 Developer Agent competition](https://www.kaggle.com/competitions/gemma-4-developer-agent).
2. Select **GPU L4 ×4** in notebook settings. The benchmark defaults to four L4 GPUs and tensor parallelism of four.
3. Attach the competition data, the **Gemma 4 Developer Agent Wheelhouse** dataset, and **Google Gemma 4 → Other → gemma-4-e4b-it-qat-w4a16-ct → Version 2**. The preflight cell prints the exact input paths.
4. Run all cells in a fresh session. The notebook uses the attached wheelhouse and checkpoint offline.

The notebook writes a CSV of test outcomes, generated patch files, and a baseline submission.zip. The practice score is for this fixed sample only. The ZIP is prepared as a notebook output for manual submission.

Adapted from the supplied mirror of Kaggle's [Getting Started notebook](https://www.kaggle.com/code/ryanholbrook/getting-started-gemma-4-developer-agent).


## 1. Choose the practice tasks and time limits

The default runs one issue each from FastAPI, Rich, and Requests in the order shown below. Within each repository it uses the first row in the attached public task file. Each run prints and records the selected IDs. For paired runs, paste those same IDs into TASK_IDS. Set PRACTICE_REPOS to just one repository for a quicker setup check, or supply up to five explicit task IDs for a larger pilot.


In [ ]:
import csv
import re
import subprocess
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
DATA_DIR = INPUT_DIR / "competitions/gemma-4-developer-agent"
WHEELHOUSE_DIR = INPUT_DIR / "datasets/metric/gemma-4-developer-agent-wheelhouse"
MODEL_NAME = "gemma-4-e4b-it-qat-w4a16-ct"
MODEL_VERSION = "2"
MODEL_PATH = INPUT_DIR / "models/google/gemma-4/other" / MODEL_NAME / MODEL_VERSION

PRACTICE_REPOS = ("fastapi/fastapi", "Textualize/rich", "psf/requests")
TASK_IDS = ()  # Example override: ("fastapi_15661", "fastapi_15588")
TENSOR_PARALLEL_SIZE = 4
MAX_MODEL_LEN = 32768
EVAL_BUDGET = {
    "timeout_seconds": 90,
    "max_tool_calls": 40,
    "max_time_minutes": 5,
    "max_turns": 60,
}
VERIFICATION_TIMEOUT_SECONDS = 300

required_inputs = {
    "competition task data": DATA_DIR / "tasks.jsonl",
    "competition sample agent": DATA_DIR / "sample_submission/agent.yaml",
    "competition snapshots": DATA_DIR / "snapshots",
    "competition graphs": DATA_DIR / "graphs",
    "competition embeddings": DATA_DIR / "embeddings",
    "repository dependency wheels": DATA_DIR / "wheels",
    "metric wheelhouse": WHEELHOUSE_DIR,
    f"E4B model version {MODEL_VERSION}": MODEL_PATH / "config.json",
}
missing = [f"{name}: {path}" for name, path in required_inputs.items() if not path.exists()]
if not list(WHEELHOUSE_DIR.glob("*.whl")):
    missing.append(f"wheel files: {WHEELHOUSE_DIR}/*.whl")
if missing:
    raise FileNotFoundError("Attach the Kaggle inputs below or update their paths.\n" + "\n".join(missing))

try:
    gpu_probe = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
        check=True, capture_output=True, text=True,
    )
except (FileNotFoundError, subprocess.CalledProcessError) as error:
    raise RuntimeError("Enable the GPU L4 ×4 accelerator and restart the Kaggle session.") from error

gpu_rows = list(csv.reader(gpu_probe.stdout.splitlines(), skipinitialspace=True))
if len(gpu_rows) < TENSOR_PARALLEL_SIZE or any("L4" not in row[0].split() for row in gpu_rows):
    raise RuntimeError(f"This notebook targets L4 GPUs. Select GPU L4 ×4. Detected: {gpu_rows}")

WORKING_DIR.mkdir(parents=True, exist_ok=True)
for name, memory_mib in gpu_rows:
    print(f"{name} · {memory_mib} MiB")
print(f"Competition checkpoint: {MODEL_PATH}")


## 2. Install the competition runtime

This uses the wheelhouse installation from the starter notebook. Start from a fresh kernel so the installed CUDA and vLLM libraries are imported together.


In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
from tempfile import mkdtemp

tmp_whl = Path(mkdtemp(prefix='e4b-wheelhouse-'))
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

# The wheelhouse ships cp312-only native wheels and relies on the image's torch/CUDA.
print(f'Python {sys.version.split()[0]} · image must match the competition starter')
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f'Wheelhouse requires CPython 3.12, found {sys.version.split()[0]}. '
        'Pin the notebook to the Getting Started notebook\'s Docker image.'
    )

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')


## 3. Prepare the untuned E4B agent

Copy Kaggle's sample agent and set every agent model declaration to E4B. The sample's 31B LoRA references and adapter files are removed before evaluation. The sample prompts, tools, and sampling settings provide the baseline. The task budget above is also written to the submission configuration.


In [ ]:
import json
import shutil

import yaml
from adk_submission.discovery import discover_declared_models
from swegemma.models import load_tasks

TASKS_PATH = DATA_DIR / "tasks.jsonl"
AGENT_DIR = WORKING_DIR / "e4b_baseline_agent"
OUTPUT_DIR = WORKING_DIR / "e4b_baseline_results"
OUTPUT_DIR.mkdir(exist_ok=True)

if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(DATA_DIR / "sample_submission", AGENT_DIR)

for yaml_path in sorted(AGENT_DIR.rglob("*")):
    if not yaml_path.is_file() or yaml_path.suffix.lower() not in {".yaml", ".yml"}:
        continue
    config_text = yaml_path.read_text(encoding="utf-8")
    config_text = re.sub(r"^([ \t]*model:[ \t]*).+$", rf"\g<1>{MODEL_NAME}", config_text, flags=re.MULTILINE)
    config_text = re.sub(r"^[ \t]*adapter:[^\n]*\n?", "", config_text, flags=re.MULTILINE)
    yaml_path.write_text(config_text, encoding="utf-8")

shutil.rmtree(AGENT_DIR / "adapters", ignore_errors=True)
(AGENT_DIR / "eval_config.yml").unlink(missing_ok=True)
(AGENT_DIR / "eval_config.yaml").write_text(
    yaml.safe_dump({"evaluation": EVAL_BUDGET}, sort_keys=False), encoding="utf-8"
)

declared_models = set(discover_declared_models(AGENT_DIR))
if declared_models != {MODEL_NAME}:
    raise ValueError(f"Expected only {MODEL_NAME}, found {sorted(declared_models)}")

tasks = load_tasks(TASKS_PATH)
tasks_by_id = {task.instance_id: task for task in tasks}
if TASK_IDS:
    missing_ids = [task_id for task_id in TASK_IDS if task_id not in tasks_by_id]
    if missing_ids:
        raise ValueError(f"Unknown public task IDs: {missing_ids}")
    sample_tasks = [tasks_by_id[task_id] for task_id in TASK_IDS]
else:
    first_by_repo = {}
    for task in tasks:
        first_by_repo.setdefault(task.repo.casefold(), task)
    missing_repos = [repo for repo in PRACTICE_REPOS if repo.casefold() not in first_by_repo]
    if missing_repos:
        raise ValueError(f"No public practice tasks for: {missing_repos}")
    sample_tasks = [first_by_repo[repo.casefold()] for repo in PRACTICE_REPOS]

selected_ids = [task.instance_id for task in sample_tasks]
if not selected_ids or len(selected_ids) != len(set(selected_ids)):
    raise ValueError("Select a nonempty list of unique practice tasks.")

run_settings = {
    "model": MODEL_NAME,
    "model_version": MODEL_VERSION,
    "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
    "max_model_len": MAX_MODEL_LEN,
    "eval_budget": EVAL_BUDGET,
    "verification_timeout_seconds": VERIFICATION_TIMEOUT_SECONDS,
    "task_ids": selected_ids,
}
(OUTPUT_DIR / "run_settings.json").write_text(
    json.dumps(run_settings, indent=2) + "\n", encoding="utf-8"
)
for task in sample_tasks:
    print(f"{task.instance_id} · {task.repo} @ {task.base_commit[:8]}")


## 4. Start vLLM and run both evaluation phases

The local notebook uses the competition starter's subprocess sandbox backend. Each public task goes through patch generation and independent verification. The CSV distinguishes passing repairs, failing repairs, missing patches, and evaluator exceptions. Each completed row is saved immediately. If the server fails to start, inspect the startup error and verify the model snapshot, wheelhouse version, and accelerator before rerunning.


In [ ]:
import time
import traceback

import litellm
import pandas as pd
import torch
from IPython.display import display
from adk_submission import VllmConfig, VllmServer, discover_adapters
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS, EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

litellm.drop_params = True
if not torch.cuda.is_available() or torch.cuda.device_count() < TENSOR_PARALLEL_SIZE:
    raise RuntimeError("PyTorch cannot see all requested L4 GPUs. Check the wheelhouse and restart.")

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
if getattr(adapters, "adapters", None):
    raise ValueError("The untuned baseline must have no adapters.")

limits, generation_constraints = build_submission_limits()
server = VllmServer(
    VllmConfig(
        model=str(MODEL_PATH),
        port=8000,
        host="127.0.0.1",
        tool_call_parser="gemma4",
        reasoning_parser="gemma4",
        max_model_len=MAX_MODEL_LEN,
        dtype="bfloat16",
        gpu_memory_utilization=0.90,
        enable_auto_tool_choice=True,
        enable_lora=False,
        max_loras=1,
        max_lora_rank=128,
        scheduler_reserve_full_isl=False,
        tensor_parallel_size=TENSOR_PARALLEL_SIZE,
        startup_timeout=1200,
        extra_args=(),
    ),
    adapter_manifest=adapters,
)

results_path = OUTPUT_DIR / "baseline_results.csv"
patches_dir = OUTPUT_DIR / "patches"
patches_dir.mkdir(exist_ok=True)
for prior_patch in patches_dir.glob("*.diff"):
    prior_patch.unlink()
results_path.unlink(missing_ok=True)
rows = []

try:
    server.start()
    print(f"vLLM ready at {server.base_url}")
    models = server.create_model_registry(
        aliases=[MODEL_NAME],
        model_prefix="openai/",
        api_key=os.environ.get("LOCAL_API_KEY", "EMPTY"),
    )
    evaluator = Evaluator(
        EvalConfig(
            tasks_path=TASKS_PATH,
            snapshots_dir=DATA_DIR / "snapshots",
            results_dir=OUTPUT_DIR / "harness",
            submission_dir=AGENT_DIR,
            models=models,
            sandbox="subprocess",
            verification_timeout_seconds=VERIFICATION_TIMEOUT_SECONDS,
            **EVAL_BUDGET,
            limits=limits,
            generation_constraints=generation_constraints,
            adapter_manifest=adapters,
            context_cache_config=ContextCacheConfig(
                min_tokens=2048, ttl_seconds=1800, cache_intervals=10
            ),
            events_compaction_config=EventsCompactionConfig(
                compaction_interval=15,
                overlap_size=2,
                token_threshold=14336,
                event_retention_size=5,
            ),
            graph_dir=str(DATA_DIR / "graphs"),
            embeddings_dir=str(DATA_DIR / "embeddings"),
            wheels_dir=DATA_DIR / "wheels",
            verbose=False,
        )
    )

    for idx, task in enumerate(sample_tasks, start=1):
        task_id = task.instance_id
        print(f"[{idx}/{len(sample_tasks)}] {task_id} ({task.repo})", flush=True)
        started = time.monotonic()
        row = {
            "task_id": task_id,
            "repo": task.repo,
            "base_commit": task.base_commit,
            "resolved": None,
            "patch_chars": None,
            "test_exit_code": None,
            "tool_calls": None,
            "harness_seconds": None,
            "detail": "",
        }
        try:
            result = await evaluator.evaluate_task(
                task=task, task_index=idx, total_tasks=len(sample_tasks)
            )
        except Exception as error:
            row.update(status="EVALUATOR_ERROR", detail=f"{type(error).__name__}: {error}")
            traceback.print_exc()
        else:
            patch = result.agent_patch or ""
            has_patch = bool(patch.strip())
            resolved = bool(result.resolved)
            if has_patch:
                (patches_dir / f"{task_id}.diff").write_text(patch, encoding="utf-8")
            detail = getattr(result, "error", None)
            row.update(
                resolved=resolved,
                status="PASS" if resolved else "NOT_RESOLVED" if has_patch else "NO_PATCH",
                patch_chars=len(patch) if has_patch else 0,
                test_exit_code=result.test_exit_code,
                tool_calls=result.tool_calls,
                harness_seconds=round(result.duration_seconds, 2),
                detail=str(detail) if detail else "",
            )
        row["wall_seconds"] = round(time.monotonic() - started, 2)
        rows.append(row)
        pd.DataFrame(rows).to_csv(results_path, index=False)
        print(f"  {row['status']} · patch chars {row['patch_chars']} · {row['wall_seconds']:.1f}s")
finally:
    try:
        server.stop()
    except Exception as cleanup_error:
        print(f"vLLM cleanup reported: {cleanup_error}")

results = pd.DataFrame(rows)
display(results)
passed = int(results["status"].eq("PASS").sum())
evaluator_errors = int(results["status"].eq("EVALUATOR_ERROR").sum())
with_patch = int(results["patch_chars"].fillna(0).gt(0).sum())
print(f"Practice result: {passed}/{len(sample_tasks)} resolved ({passed / len(sample_tasks):.1%})")
print(f"Extracted patches: {with_patch} · evaluator exceptions: {evaluator_errors}")
if evaluator_errors:
    print("Inspect evaluator exceptions before comparing this practice result with another run.")
print(f"CSV: {results_path}")


## 5. Package the baseline agent

The archive contains the agent configuration and the files it references. The CSV and practice patches are separate notebook outputs. The ZIP is available in the Kaggle notebook's Output tab after the notebook version finishes.


In [ ]:
import zipfile

from adk_submission import validate_directory
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

limits, _ = build_submission_limits()
for path in AGENT_DIR.rglob("*"):
    if path.is_symlink():
        raise ValueError(f"Symlink inside submission directory: {path.relative_to(AGENT_DIR)}")
validate_directory(AGENT_DIR, limits=limits)

archive_files = sorted(path for path in AGENT_DIR.rglob("*") if path.is_file())
total_bytes = sum(path.stat().st_size for path in archive_files)
if total_bytes >= MAX_SUBMISSION_SIZE_BYTES:
    raise ValueError(f"Submission exceeds the unpacked archive limit: {total_bytes:,} bytes")
for path in archive_files:
    if path.suffix.lower() not in ALLOWED_SUBMISSION_EXTENSIONS:
        raise ValueError(f"Unsupported archive file: {path.relative_to(AGENT_DIR)}")

zip_path = WORKING_DIR / "submission.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in archive_files:
        archive.write(path, arcname=path.relative_to(AGENT_DIR).as_posix())
with zipfile.ZipFile(zip_path) as archive:
    if "agent.yaml" not in archive.namelist() or archive.testzip() is not None:
        raise ValueError("Archive verification failed.")
print(f"Created {zip_path} ({zip_path.stat().st_size:,} bytes)")
